# RAG with QueryChat: NYC Taxi Demo

**The problem querychat solves without RAG:** the LLM knows the column *names* and
value *ranges* from the data schema, but not what they *mean*. We can use data description in querychat to put the additional description in systems prompt, but what if it is huge and odd (as almost any rulebook)?
Well, ideally we would want to inject only the portion relevant to each query with local information, and for add it to the query. This is where Retrieval-Augmented Generation knocks to our doors.

In our example, LLM sees `RatecodeID` values range from 1–6, but doesn't know that 2 = JFK flat rate.

**RAG fixes this:** for every user question, retrieve the relevant domain knowledge chunks
and inject them into the user message before it reaches the LLM.
Different questions -> different chunks -> the right (hopefully, search is not perfect, and knowledge base can be incomplete) context every time.

```mermaid
flowchart LR
    Q([User question]) --> R[TF-IDF retrieve<br/>top-k chunks]
    KB[(Knowledge<br/>Base)] --> R
    R --> A[Augmented<br/>message]
    A --> LLM[LLM]
    LLM --> SQL[SQL]
    SQL --> DB[(DuckDB)]
    DB --> Ans([Answer])
```

**What we'll build:**

1. `QueryChat` without RAG: see what the LLM knows from schema alone
2. TF-IDF knowledge base: retrieve relevant chunks for any query
3. Per-query injection: augment the user message
4. Side-by-side: same questions with vs without domain knowledge
5. Tool loop walkthrough
6. How to wire this into a Shiny app

In [1]:
import os
import chatlas as ctl
import pandas as pd
from pathlib import Path
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from dotenv import load_dotenv, find_dotenv
from querychat import QueryChat

# Load .env — try cwd, cwd/.., and cwd/rag/ to handle running from repo root or rag/
_cwd = Path.cwd()
for _p in [_cwd / ".env", _cwd / "rag" / ".env", _cwd.parent / ".env"]:
    if _p.exists():
        load_dotenv(_p)
        break

---
## A. QueryChat without RAG

Let's see what the LLM knows *before* we inject any domain knowledge.
We sample 50 000 rows — enough for meaningful queries, fast for DuckDB.

In [2]:
taxi = pd.read_parquet("../data/taxi_2024-01.parquet").sample(50_000, random_state=42)
print(f"Loaded {len(taxi):,} rows × {len(taxi.columns)} columns")

Loaded 50,000 rows × 19 columns


In [3]:
qc_base = QueryChat(
    taxi,
    "taxi",
    client=ctl.ChatGithub(model="gpt-4.1-mini", api_key=os.getenv("GITHUB_TOKEN")),
)

In [4]:
print(qc_base.system_prompt)

You are a data dashboard chatbot that operates in a sidebar interface. Your role is to help users interact with their data through filtering, sorting, and answering questions.

You have access to a DuckDB SQL database with the following schema:

<database_schema>
Table: taxi
Columns:
- VendorID (INTEGER)
  Range: 1 to 6
- tpep_pickup_datetime (TIME)
  Range: 2023-12-31 23:58:35 to 2024-01-31 23:56:27
- tpep_dropoff_datetime (TIME)
  Range: 2024-01-01 00:10:55 to 2024-02-01 07:56:40
- passenger_count (FLOAT)
  Range: 0.0 to 8.0
- trip_distance (FLOAT)
  Range: 0.0 to 56.22
- RatecodeID (FLOAT)
  Range: 1.0 to 99.0
- store_and_fwd_flag (TEXT)
  Categorical values: 'N', 'Y'
- PULocationID (INTEGER)
  Range: 1 to 265
- DOLocationID (INTEGER)
  Range: 1 to 265
- payment_type (INTEGER)
  Range: 0 to 4
- fare_amount (FLOAT)
  Range: -170.0 to 577.96
- extra (FLOAT)
  Range: -7.5 to 11.75
- mta_tax (FLOAT)
  Range: -0.5 to 0.5
- tip_amount (FLOAT)
  Range: -1.0 to 422.7
- tolls_amount (FLOAT)


**Notice:**
- The `<database_schema>` block lists column names, types, and value *ranges*
  (e.g. `RatecodeID`: 1.0–99.0), but no meaning for those codes.
- There's no `<data_description>` block, it only appears when you pass one.
- The LLM will have to guess that `RatecodeID=2` might be airport-related.

Let's confirm: ask a question that requires domain knowledge.

In [5]:
_client = qc_base.client()
_response = _client.chat(
    "How do I filter for JFK airport trips? What column and value should I use?",
    echo="none",
)
print("WITHOUT RAG:", _response)

RateLimitError: Too many requests. For more on scraping GitHub and how it may affect your rights, please review our Terms of Service (https://docs.github.com/en/site-policy/github-terms/github-terms-of-service).

---
## B. Building the Knowledge Base

Our KB is plain `.txt` — paragraphs separated by blank lines, one topic per paragraph.
This is the simplest possible RAG KB format.

**TF-IDF retrieval:** no API key needed, no model download.
It works on term overlap — good enough for structured domain glossaries.
(For semantic similarity across natural language, you'd use sentence-transformers or an embedding API.)

In [ ]:
_kb_path = Path("../rag/knowledge_base/taxi_glossary.txt")
kb_chunks = [c.strip() for c in _kb_path.read_text().split("\n\n") if c.strip()]
kb_vectorizer = TfidfVectorizer()
kb_vectors = kb_vectorizer.fit_transform(kb_chunks)
print(f"Loaded {len(kb_chunks)} chunks → TF-IDF matrix shape: {kb_vectors.shape}")

In [ ]:
def retrieve(query: str, top_k: int = 3) -> list[str]:
    """Return top_k most relevant chunks for query using TF-IDF cosine similarity."""
    q_vec = kb_vectorizer.transform([query])
    scores = cosine_similarity(q_vec, kb_vectors).flatten()
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [kb_chunks[i] for i in top_idx if scores[i] > 0]

print("Query: 'how are airport trips identified?'")
for _c in retrieve("how are airport trips identified?"):
    print(" ·", _c[:80])

**TF-IDF works well for structured glossaries**, meaning exact term matches, no API key, runs in < 1ms.

When to upgrade to **embeddings** (semantic similarity, handles paraphrases and synonyms):

**`sentence-transformers` (local):**
- Free, no API key
- `pip install sentence-transformers`, model [`all-MiniLM-L6-v2`](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2)
- ~5ms on CPU — good for local demos

**OpenAI Embeddings API:**
- ~0.00002 USD / 1K tokens, API key required
- Model [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings)
- ~100ms network — better for production / large KBs

**TF-IDF works well for structured glossaries**, meaning exact term matches, no API key, runs in < 1ms.

When to upgrade to **embeddings** (semantic similarity, handles paraphrases and synonyms):

| | `sentence-transformers` | OpenAI Embeddings API |
|---|---|---|
| Cost | Free, runs locally | ~0.00002 USD / 1K tokens |
| Setup | `pip install sentence-transformers` | API key required |
| Model | [all-MiniLM-L6-v2](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) | [text-embedding-3-small](https://platform.openai.com/docs/guides/embeddings) |
| Speed | ~5ms CPU | ~100ms network |
| When | Local demo, no API budget | Production, large KB |

---
## C. Per-query RAG injection

The trick: inject retrieved context into the **user message**.
This way every question gets its own relevant chunks — true per-query retrieval.

```python
def chat_with_rag(client, query):
    chunks = retrieve(query, top_k=3)
    if chunks:
        context = "\\n\\n".join(chunks)
        query = f"Relevant domain context:\\n{context}\\n\\nQuestion: {query}"
    return client.chat(query, echo="none")
```

`QueryChat` is built once with no `data_description` — it stays generic.
Retrieval happens fresh for every `.chat()` call.

In [ ]:
qc_rag = QueryChat(
    taxi,
    "taxi",
    client=ctl.ChatGithub(model="gpt-4.1-mini", api_key=os.getenv("GITHUB_TOKEN")),
)

In [ ]:
def chat_with_rag(client, query: str) -> str:
    chunks = retrieve(query, top_k=3)
    if chunks:
        context = "\n\n".join(chunks)
        query = f"Relevant domain context:\n{context}\n\nQuestion: {query}"
    return client.chat(query, echo="none")

---
## D. Side-by-side: queries that need domain knowledge

These queries require knowing what the *codes* mean: exactly where RAG helps.

**With vs without:**

```mermaid
flowchart TD
    subgraph with[With RAG]
        direction LR
        Q2([filter JFK trips?]) --> Ret[retrieve:<br/>RatecodeID 2 = JFK]
        Ret --> LLM2[LLM]
        LLM2 --> C(["RatecodeID = 2 ✓"])
    end
    subgraph without[Without RAG]
        direction LR
        Q1([filter JFK trips?]) --> LLM1[LLM]
        LLM1 --> G(["RatecodeID = 99 ❌"])
    end

```

In [ ]:
Q1 = [
    "How do I filter for JFK airport trips? What column and value should I use?",
]

for _q in Q1:
    print(f"❓ {_q}")
    print("  WITHOUT RAG:", qc_base.client().chat(_q, echo="none"))
    print("-" * 80)
    print("  WITH RAG:   ", chat_with_rag(qc_rag.client(), _q))
    print("=" * 80)

In [ ]:
Q2 = [
    "What share of trips used a negotiated fare rate?",
]

for _q in Q2:
    print(f"❓ {_q}")
    print("  WITHOUT RAG:", qc_base.client().chat(_q, echo="none"))
    print("-" * 80)
    print("  WITH RAG:   ", chat_with_rag(qc_rag.client(), _q))
    print("=" * 80)

In [ ]:
Q3 = [
    "Compare average tip for credit card vs cash payments — which columns encode this?",
]

for _q in Q3:
    print(f"❓ {_q}")
    print("  WITHOUT RAG:", qc_base.client().chat(_q, echo="none"))
    print("-" * 80)
    print("  WITH RAG:   ", chat_with_rag(qc_rag.client(), _q))
    print("=" * 80)

---
## E. Watching the tool loop

Same as the querychat notebook from the last lecture. The augmented query is what the LLM sees; the tool loop is unchanged.

In [ ]:
_step = [0]

def _on_req(req):
    _step[0] += 1
    print(f"── Step {_step[0]}: LLM requests tool ──")
    print(f"   Tool: {req.name}  Args: {req.arguments}")

def _on_res(res):
    _step[0] += 1
    print(f"── Step {_step[0]}: Tool returns result ──")
    print(f"   {str(res.value)[:300]}")

_client = qc_rag.client()
_client.on_tool_request(_on_req)
_client.on_tool_result(_on_res)

_query = "What is the average fare for JFK trips vs all trips?"
_chunks = retrieve(_query, top_k=3)
_augmented = f"Relevant domain context:\n{chr(10).join(_chunks)}\n\nQuestion: {_query}" if _chunks else _query

print("── Step 0: User sends message ──────────────────────")
print("   Original query:", _query)
print("   Augmented with RAG context:")
print("   " + _augmented[:400].replace("\n", "\n   "))
print()
_response = _client.chat(_augmented, echo="none")
_step[0] += 1
print(f"── Step {_step[0]}: LLM final response ──")
print(_response)

---
## F. Wiring into a Shiny App

Everything you saw maps directly to a Shiny app:

- `retrieve()` + TF-IDF index **module level** (built once, shared across sessions)
- `QueryChat(...)` → **module level** too — no `data_description`, stays generic
- `chat_with_rag(client, query)` wraps the per-message send inside the reactive effect

```python
# Module level
qc = QueryChat(df, "taxi", client=ChatGithub(...))

def server(input, output, session):
    chat_session = qc.client()

    @reactive.effect
    @reactive.event(input.querychat_user_input)
    async def _():
        query = input.querychat_user_input()
        chunks = retrieve(query, top_k=3)
        if chunks:
            augmented = f"Relevant context:\\n{'\\n\\n'.join(chunks)}\\n\\nQuestion: {query}"
            chat_session.chat(augmented, echo="none")
```

`retrieve()` is fast (TF-IDF, no API) — adds < 1ms per query.